In [386]:
import pandas as pd
import numpy as np
from collections import defaultdict
from itertools import combinations
import time


df = pd.read_csv("market.csv", sep=";", encoding="utf-8-sig")

In [387]:
def dataframe_to_transactions(df):
    transactions = []
    for _, row in df.iterrows():
        items = row[row == 1].index.tolist()
        transactions.append(frozenset(items))
    return transactions

transactions = dataframe_to_transactions(df)
len(transactions)

464

Pobierz zbiór danych Real Market Data for Association Rules dostępny w Kaggle. Przyjrzyj
się danym. Spróbuj intuicyjnie / ręcznie znaleźć jakieś zbiory produktów często kupowanych razem. Co znaczy często? Jak sensownie ustawić próg liczby transakcji, w których ma wystę-
pować zbiór produktów, żeby uznać go za występujący często?

Napisz własną implementację algorytmu Apriori, omawianego na wykładzie, do znajdowania
częstych zbiorów przedmiotów. Zwróć uwagę na efektywność swojej implementacji (być może
warto użyć funkcji frozenset()).

In [388]:
transactions = []
cols = list(df.columns)

for _, row in df.iterrows():
    items = [c for c in cols if row[c] == 1]
    transactions.append(frozenset(items))

N = len(transactions)
print("Przykładowa transakcja:", transactions[0])


Przykładowa transakcja: frozenset({'Bread', 'Carrot', 'Bacon', 'Apple', 'Egg', 'Sugar', 'Hazelnut', 'HeavyCream', 'Banana'})


In [389]:
def frequent_1_itemsets(transactions, min_support):
    counts = defaultdict(int)

    for t in transactions:
        for item in t:
            counts[frozenset([item])] += 1

    return {item: cnt for item, cnt in counts.items() if cnt >= min_support}


In [390]:
def generate_candidates(prev_itemsets, k):
    prev_itemsets = list(prev_itemsets)
    candidates = set()

    for i in range(len(prev_itemsets)):
        for j in range(i + 1, len(prev_itemsets)):
            c = prev_itemsets[i] | prev_itemsets[j]
            if len(c) == k:
                candidates.add(c)

    return candidates


In [391]:
def count_support(candidates, transactions, min_support):
    counts = defaultdict(int)

    for t in transactions:
        for c in candidates:
            if c.issubset(t):
                counts[c] += 1

    return {c: cnt for c, cnt in counts.items() if cnt >= min_support}


In [392]:
def apriori(transactions, min_support_rel=0.01):
    N = len(transactions)
    min_support = max(1, int(min_support_rel * N))

    Lk = frequent_1_itemsets(transactions, min_support)
    frequent_itemsets = dict(Lk)
    k = 2

    while Lk:
        candidates = generate_candidates(Lk.keys(), k)
        if not candidates:
            break

        Lk = count_support(candidates, transactions, min_support)
        frequent_itemsets.update(Lk)
        k += 1

    return frequent_itemsets, N


In [393]:
frequent_itemsets, N = apriori(transactions, min_support_rel=0.03)

print("Liczba częstych zbiorów:", len(frequent_itemsets))

sizes = {}
for s in frequent_itemsets:
    sizes[len(s)] = sizes.get(len(s), 0) + 1

print("Rozmiary zbiorów:", sizes)


Liczba częstych zbiorów: 15639
Rozmiary zbiorów: {1: 22, 2: 231, 3: 1540, 4: 7099, 5: 6370, 6: 374, 7: 3}


3. Stwórz reguły asocjacyjne z otrzymanych częstych zbiorów przedmiotów. Jak sensownie ustawić
progi support i confidence dla reguł asocjacyjnych?

4. Przejrzyj otrzymane reguły asocjacyjne. Jak wybrać z nich te najbardziej wartościowe? Może warto użyć lift lub leverage? Może masz pomysły na inne metryki oceniające reguły asocjacyjne?

In [394]:
def generate_rules(frequent_itemsets, N,
                   min_confidence=0.3,
                   min_support_rel=0.01):

    support_rel = {s: cnt / N for s, cnt in frequent_itemsets.items()}
    rules = []

    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue

        sup_I = support_rel[itemset]

        for r in range(1, len(itemset)):
            for A in combinations(itemset, r):
                A = frozenset(A)
                B = itemset - A

                if A not in support_rel or B not in support_rel:
                    continue

                confidence = sup_I / support_rel[A]

                if confidence < min_confidence or sup_I < min_support_rel:
                    continue

                lift = sup_I / (support_rel[A] * support_rel[B])
                leverage = sup_I - support_rel[A] * support_rel[B]

                rules.append({
                    "A": A,
                    "B": B,
                    "support": sup_I,
                    "confidence": confidence,
                    "lift": lift,
                    "leverage": leverage
                })

    return rules


In [395]:
rules = generate_rules(
    frequent_itemsets, N,
    min_confidence=0.3,
    min_support_rel=0.03
)

print("Liczba reguł:", len(rules))


Liczba reguł: 100347


In [396]:
rules_sorted = sorted(rules, key=lambda r: r["lift"], reverse=True)

for r in rules_sorted[:10]:
    print(
        r["A"], "→", r["B"],
        "| support:", round(r["support"], 3),
        "| conf:", round(r["confidence"], 3),
        "| lift:", round(r["lift"], 3)
    )


frozenset({'Cucumber', 'Egg', 'ShavingFoam'}) → frozenset({'Salt', 'Flour', 'Banana'}) | support: 0.03 | conf: 0.326 | lift: 4.443
frozenset({'Salt', 'Flour', 'Banana'}) → frozenset({'Cucumber', 'Egg', 'ShavingFoam'}) | support: 0.03 | conf: 0.412 | lift: 4.443
frozenset({'Carrot', 'Shampoo', 'Bacon'}) → frozenset({'Meat', 'Egg', 'Honey'}) | support: 0.037 | conf: 0.472 | lift: 4.382
frozenset({'Meat', 'Egg', 'Honey'}) → frozenset({'Carrot', 'Shampoo', 'Bacon'}) | support: 0.037 | conf: 0.34 | lift: 4.382
frozenset({'Butter', 'Carrot', 'Meat'}) → frozenset({'Cheese', 'Onion', 'Bacon'}) | support: 0.03 | conf: 0.452 | lift: 4.276
frozenset({'Salt', 'Flour', 'Cucumber', 'Banana'}) → frozenset({'Egg', 'ShavingFoam'}) | support: 0.03 | conf: 0.778 | lift: 4.246
frozenset({'Onion', 'Carrot', 'Meat'}) → frozenset({'Egg', 'Bacon', 'Honey'}) | support: 0.041 | conf: 0.475 | lift: 4.238
frozenset({'Egg', 'Bacon', 'Honey'}) → frozenset({'Onion', 'Carrot', 'Meat'}) | support: 0.041 | conf: 0.365 

Zwiększ ilość danych dopisując milion nowych transakcji według własnego pomysłu (możesz na
przykład duplikować istniejące transakcje dopisując lub usuwając z nich kilka losowych przed-
miotów). Uruchom na nich swoją implementację algorytmu Apriori, zmierz czas działania,
przemyśl czy implementacja jest efektywna, przejrzyj wyniki. W razie potrzeby (ekstremal-
nie długiego czasu działania, ograniczeń sprzętowych, itp.) możesz zmniejszyć liczbę nowych
transakcji.

In [397]:
def add_transactions(df, extra=1_000_000, flip_prob=0.05):
    values = df.values
    n_rows, n_cols = values.shape
    new_rows = []

    for _ in range(extra):
        base = values[np.random.randint(0, n_rows)].copy()
        flip = np.random.rand(n_cols) < flip_prob
        base[flip] = 1 - base[flip]
        new_rows.append(base)

    df_big = pd.DataFrame(new_rows, columns=df.columns)
    return pd.concat([df, df_big], ignore_index=True)


In [ ]:
df_big = add_transactions(df, extra=1_000_000)
transactions_big = dataframe_to_transactions(df_big)

start = time.time()
frequent_big, N_big = apriori(transactions_big, min_support_rel=0.03)
end = time.time()

print("Czas Apriori:", end - start) #to sie nie policzy nigdy 


KeyboardInterrupt: 

6. Powtórz poprzedni punkt zwiększając liczbę produktów o tysiąc według własnego pomysłu
(duplikując transakcje możesz zamieniać występujące w nich produkty na nowe).

In [ ]:
def add_products(df, n_new=1000):
    df_ext = df.copy()
    for i in range(n_new):
        df_ext[f"NewProduct_{i}"] = np.random.binomial(1, 0.01, len(df))
    return df_ext

In [ ]:
df_more_products = add_products(df_big, n_new=1000)
transactions_more_products = dataframe_to_transactions(df_more_products)

start = time.time()
frequent_more, N_more = apriori(transactions_more_products, min_support_rel=0.03)
end = time.time()

print("Czas Apriori:", end - start)


KeyboardInterrupt: 